# Transformers Overview


### Introduction
Transformers, in their original formulation, were models that were used to translate text from one language to another. However, they have been increasingly used for the general type of model to process text (and even other modalities, but this is out of the scope of this document).


### Tokenization
Since it is difficult to use mathematical models to work on raw textual data, we **tokenize** the text (break the text into chunks, and each unique chunk of text is mapped to an integer which will be consistent throughout the dataset), then map each token to a vector **embedding**, thus we get a sequence of vectors, or a matrix. Each row of this matrix will represent the **embedding** of each token.


### Embeddings
**Embeddings** are vectors which can encode the semantics of a given token. Suppose we have a token "elephant", and we want to represent the qualities of this token as a vector. We can think of each position in the vector as a **feature**, or a number that encodes a specific quality of the token. If we had 2 features, which, for demonstration, we choose **size** and **intelligence**, qualities that can both be represented as a number, we can then encode the token "elephant" as a vector with relatively high values for both features (as elephants are indeed big and intelligent). We can do the same for other tokens that can be described with these features. Of course this is a simplification, and most models do not have these very interpretable features that can describe all words.


### Positional Encoding
After creating this matrix, we add the positional information of each token to each embedding. This is necessary for the model, as the operations that transformers perform are generally agnostic to the position of each token (i.e. whether a word is close to the start or end of a sentence). We do this by applying a function on each row of the matrix, that depends on its position and (sometimes) the row itself. There are multiple methods to apply positional information to each embedding, and this will be covered later in this document.


### Transformer Blocks
The processed matrix will now be fed to stacks of **transformer blocks**. Transformer blocks, on a high level, allow each token to attend to each other through an operation called **attention**. This roughly means that it will be able to understand how the semantics of words in a sentence depend on their context. The output of each transformer block is a matrix of the same size as the input, where each row reflects a better semantic encoding of its respective token. This output can then be directly fed into the next block. Transformer blocks can be divided roughly into two types: **encoder** blocks and **decoder** blocks. The most significant difference between the two is that in encoder blocks, each token can attend to all other tokens in the sequence, while in decoder blocks, tokens can only attend to tokens that come before the respective token. This should make sense, as encoders only need to "read" text, and return a matrix that represents the semantics of the text, while decoders produce text, and therefore they cannot see the future (text that it has to produce).




### Applications: Translation, Text Generation, Semantic Encoding
After the matrix is fed through the transformer blocks, there are a couple of possibilities:


In the original transformer paper *Attention is all you need*, their model used an encoder-decoder structure, where the encoder takes in and processes the input text, while the decoder uses the processed input text to produce the translated text. The matrix is put through the stack of encoder blocks (our previous step), then the output is read using cross attention by a stack of decoder blocks, and at the end of the decoder stack, a simple feed forward neural network uses the embedding from the last token to produce a probability distribution of the next (translated) token. This distribution is sampled from and added to the output (which starts out empty), and the decoder takes it in again and uses both the generated (translated) text and the encoded (input) text to generate more translated tokens.


In the GPT paper, their model uses only a decoder transformer, where after the input matrix is fed through a stack of transformer blocks, a simple feed forward neural network uses the embedding from the last token to generate a probability distribution of the possible next tokens. This distribution is sampled from (which can be controlled to weight or only choose most probable outputs) and the new token is added to the text. The text with the new token is fed through the model again, which results in another token and so on and so forth, which allows for text generation. Put simply, the model keeps predicting the most probable next token and adds it to the text.


Sometimes only the embeddings of a stack of encoders are necessary for a task. This is common in tasks like encoding the semantics of a document for document retrieval in search engines. Often we only want a single vector to represent the semantics of the document, so we add a special token to the document and return the embedding of that token after feeding the whole document through the model.



# Implementation

## Library Imports
We start by importing necessary libraries to build the model, and some other tools.

In [5]:
from einops import rearrange

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from typing import Optional


## Tokenization

Given a corpus of text, we want to train a tokenizer, T, to be able to split the text into an ordered list of tokens losslessly. There are two main methods which will be implemented in this notebook:

Method 1: Character-Level Tokenization
- Each character is considered it's own token. This is the simplest method to tokenize text, and results in very long sequences, and subpar performance.

Method 2: BPE
- An algorithm that can tokenize words into subwords effectively. This will be implemented in Rust for speed.


In [6]:


class CharacterTokenizer:

    def __init__(self):
        self.char_to_idx = {}
        self.idx_to_char = {}

        for x in range(256):
            self.char_to_idx[bytes([x])] = x
            self.idx_to_char[x] = chr(x)

    def tokenize(self, text: bytes | str):
        if type(text) == str:
            text = text.encode('utf-8')
            
        tokens = torch.tensor([char for char in text], dtype=torch.long)
        return tokens
    
    def decode(self, tokens: torch.Tensor):
        text = ""
        for token in tokens:
            text += self.idx_to_char[token]
        return text


## Embedding Layer and Positional Encoding

After tokenizing the text, we get a sequence of token ids, which we must convert into a matrix of vector embeddings for each token. These embeddings will all be vectors of length $d_{model}$, which will be the dimensionality of all subsequent embeddings in the model.

In [7]:

# Each token is represented by a vector of size d_model. The embedding matrix is a matrix of size (vocab_size, d_model)
embedding = nn.Embedding(256, 512)

## Positional Encoding

Since the transformer architecture has no way of utilizing positional information by default, we must add positional information directly into the embeddings of our tokens. There are a couple of ways to encode positional embeddings:

1. Sinusoidal embeddings
- In the original paper *Attention is all you need*, the authors create vectors from a certain formula including sinusoidal functions that are then added to each embedding.
2. Learned embeddings
- Embeddings can also be learned during the training process. A matrix of size (max_seq_len, d_model) will be added to the embeddings, and will be included as parameters during optimization. This method is restrictive, as this means that positional information cannot be added to sequences longer than the maximum sequence length.
3. Rotary positional embeddings (RoPE)
- A method to encode position which has better empirical results and aligns with the intuition of dot product attention (as it is multiplicative rather than additive). Every pair of features in each embedding is treated as a pair of 2D coordinates, and is rotated in the 2D plane as the sequence progresses. Pairs are rotated by different angles depending on their position in the feature/embedding vector.

In [8]:

class LearnedPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 512):
        super().__init__()

        self.d_model = d_model
        self.pe = nn.Embedding(max_seq_len, d_model)

        # Register position indices (0 to max_seq_len - 1)
        self.register_buffer("position_ids", torch.arange(max_seq_len).unsqueeze(0))  # Shape: (1, max_seq_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: Tensor of shape (batch_size, seq_len, d_model)
        Returns: Tensor of shape (batch_size, seq_len, d_model) with positional encodings added
        """

        batch_size, seq_len, _ = x.shape  # Ensure input shape is (batch_size, seq_len, d_model)
        positions = self.position_ids[:, :seq_len]  # Shape: (1, seq_len)

        pos_embeddings = self.pe(positions)  # Shape: (1, seq_len, d_model)
        pos_embeddings = pos_embeddings.expand(batch_size, seq_len, self.d_model)  # Match batch size

        return x + pos_embeddings  # Element-wise addition


class RoPE(nn.Module):
    def __init__(self, d_k: int):
        super().__init__()
        assert d_k % 2 == 0, "d_k must be even"
        self.d_k = d_k
        
        theta = 10000 ** (-2 * (torch.arange(0, d_k, 1) // 2) / d_k)
        self.register_buffer('theta', theta)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-1] == self.d_k

        # The batch size will be the first dimension
        # The sequence length will be the second to last dimension
        batch_size, seq_len = x.shape[0], x.shape[-2]
        
        # Create y by swapping pairs in the last dimension
        y = rearrange(x, "... (d j) -> ... d j", j=2)[..., [1, 0]]
        y = rearrange(y, "... d j -> ... (d j)")
        
        # Compute angles for each position and dimension
        positions = torch.arange(seq_len, dtype=torch.float32, device=x.device)
        thetas = positions.unsqueeze(-1) * self.theta.unsqueeze(0)  # (seq_len, d_model)
        
        # Compute cos and sin terms
        cos_terms = torch.cos(thetas)
        sin_terms = torch.sin(thetas)
        
        # Apply negation to sin terms for even indices
        sin_terms[:, 0::2] *= -1
        
        # Combine using rotation formula
        rotated_x = x * cos_terms.unsqueeze(0) + y * sin_terms.unsqueeze(0)
        return rotated_x
       

        


## Transformer Blocks

### Multi-Head Attention

In transformer neural networks, the multihead attention blocks allow each token to attend to its context. Each multihead attention block takes in a tensor of size (batch_size, seq_len, d_model), X, representing the embeddings of the sequence of tokens from the previous transformer block (or the original embeddings if it's the first block), and projects them 3 times with learned weight matrices, $W_i^Q,W_i^K,W_i^V$, ($W_i^Q,W_i^K$ have a size of (d_model, d_k) and $W_i^V$ has a size of (d_model, d_v)) to Q, K, and V, respectively for each attention block (h attention blocks in total). An attention block takes in 3 matrices, Q (seq_len, d_k), K (seq_len, d_k), and V (seq_len, d_v) and outputs a matrix of size (seq_len, d_v). After each attention block is calculated, the results from each head are concatenated across the d_v axis (each head is (seq_len, d_v) so h heads concatenated will be (seq_len, d_v * h = d_model)).

**Why use multiple heads instead of just 1 attention block with d_model dimensional inputs and outputs?**

This allows the model to be able to learn multiple distinct notions of "relevance", which would not be possible with a single head, due to averaging (according to "Attention is all you need").

</br>

### Equations

#### Attention Equation
$$
\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

To break this equation down:

$\text{softmax}(\frac{QK^T}{\sqrt{d_k}})$ represents the "attention matrix", $A$.

Values in the attention matrix, $A_{ij}$, mean the "relevance" of the token at position j with respect to the token at position i. Each row, i, of the attention matrix represents the weights with which to sum all the tokens (using the value matrix) in order to get the updated token at position i. These weights are calculated by taking the dot products between all pairs of query and key embeddings, hence the matrix multiplication, dividing by a constant $\sqrt{d_k}$ for stability, and then taking the softmax rowwise to get weights that sum to 1.

Therefore, we multiply this attention matrix wth the value matrix $V$ to get our result:

$$
\text{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

#### Multihead Attention

This is multihead attention:

$$
\text{MultiHeadAttention} = \text{concat}(\text{head}_1,\text{head}_2,\text{head}_3,...,\text{head}_h)
$$

$$
\text{head}_i = \text{Attention}(XW_i^Q,XW_i^K,XW_i^V)
$$

#### Attention in Transformer Decoders
In a decoder block, we mask the future tokens before computing the softmax, like this:


$$
\text{DecoderAttention}(Q,K,V) = \text{softmax}(\frac{QK^T + \text{Mask}}{\sqrt{d_k}})V
$$

where $\text{Mask}_{ij} = \begin{cases}
-\infty & \text{if } j > i \\
0 & \text{otherwise}
\end{cases}$

</br>

Example (if seq_len = 4):
</br>
$\text{Mask} = \begin{bmatrix}
    0 & -\infty & -\infty & -\infty \\
    0 & 0 & -\infty & -\infty \\
    0 & 0 & 0 & -\infty \\
    0 & 0 & 0 & 0 \\
\end{bmatrix}$

Notes:
- For implementations of very large models, the attention is computed in a different way to reduce redundant computation.

- Instead of multi-head attention, some llms use multi-query attention, where $W_i^K$ and $W_i^V$ is shared between heads--only the weight matrices for queries are different between heads.

In [9]:

# creates an attention object, which computes the attention operation for the specific values of d_k and d_v (technically d_v is unnecesary)
class Attention(nn.Module):
    def __init__(self, d_k: int, d_v: Optional[int] = None, decoder: bool = True):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v if d_v is not None else d_k
        self.decoder = decoder
        self.scale = np.sqrt(d_k)

    def forward(self,Q: torch.Tensor, K: torch.Tensor,V: torch.Tensor):
        '''
        Arguments:
            Q: Tensor of shape (batch_size, heads, seq_len, d_k)
            K: Tensor of shape (batch_size, heads, seq_len, d_k)
            V: Tensor of shape (batch_size, heads, seq_len, d_v)
        '''


        # the K tensor has its last 2 axes transposed
        x = torch.matmul(Q, torch.transpose(K,-2,-1)) / self.scale

        if self.decoder:
            seq_len = Q.shape[-2]

            # torch.full creates a matrix of a given size filled with a given element. torch.triu takes in a matrix and returns a matrix of the same size with all but elements above the main diagonal set to zero.
            # creates the attention mask that prevents future tokens from influencing past tokens
            mask = torch.triu(torch.full((seq_len, seq_len), -np.inf), diagonal=1).to(Q.device)

            x += mask

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -1)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x


# creates the multiheaded attention block for a transformer block
# can be customized to work for encoder or decoder blocks
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, d_k: int, d_v: int, heads: int=8, decoder: bool=True, pos_enc: str="rope"):
        '''
        Arguments:
            d_model: int - the dimensionality of the model, or embeddings
            d_k: int - the dimensionality of vectors in each attention head
            d_v: int - the dimensionality of value vectors (should probably be the same as d_k)
            heads: int (default: 8) - the number of attention heads in the model
            decoder: bool (default: True) - set to True for decoder attention otherwise set to False
            pos_enc: str (default: "rope") - set to "learned" for learned positional encoding otherwise set to "rope"
        '''

        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = heads
        self.pos_enc = pos_enc
        self.pe = None
        if pos_enc == "rope":
            self.pe = RoPE(d_k)

        # can be decoder or encoder self attention
        self.attention = Attention(d_k, d_v, decoder=decoder)

        # input size should be the last dimension of the tensor, which is d_model
        self.query_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.key_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.value_weights = nn.Linear(in_features=d_model, out_features=d_v * self.h, bias=False)

        self.projection = nn.Linear(in_features=d_v * self.h, out_features=d_model)
    
    def forward(self, x: torch.Tensor):
        batch_size, seq_len, d_model = x.size() if len(x.size()) == 3 else (1,*x.size())

        # shape: (batch_size, seq_len, d_k * heads)
        queries = self.query_weights(x)
        keys = self.key_weights(x)

        # shape: (batch_size, seq_len, d_v * heads)
        values = self.value_weights(x)

        # using einops, rearrange the tensors to have shape (batch_size, heads, seq_len, d_k)
        queries = rearrange(queries, "batch seq_len (d_k heads) -> batch heads seq_len d_k", heads=self.h)
        keys = rearrange(keys, "batch seq_len (d_k heads) -> batch heads seq_len d_k", heads=self.h)
        values = rearrange(values, "batch seq_len (d_v heads) -> batch heads seq_len d_v", heads=self.h)
        
        # apply positional encoding if it exists (for RoPE)
        if self.pe is not None:
            queries = self.pe(queries)
            keys = self.pe(keys)

        # size will be (batch_size, heads, seq_len, d_v)
        results = self.attention(queries, keys, values)

        results = rearrange(results, "batch heads seq_len d_v -> batch seq_len (d_v heads)")
        results = self.projection(results)
        return results



**Testing attention outputs:**

Here we create an attention block that handles vectors of $d_k = 8$ and $d_v = 8$, respectively. The output shows the result of attention on the 3 randomly generated matrices.

In [10]:
block = Attention(8,8)
block.forward(torch.randn((9,8)),torch.randn((9,8)),torch.randn((9,8)))

tensor([[-1.1888e+00,  1.2600e+00, -7.2582e-01, -8.2776e-01,  1.0251e+00,
         -5.4721e-01,  3.4832e+00,  8.3838e-01],
        [-1.1970e+00,  1.0555e+00, -5.5235e-01, -7.0437e-01,  1.0803e+00,
         -4.4633e-01,  3.2013e+00,  6.5658e-01],
        [-1.0462e+00,  3.5549e-01,  3.6204e-02, -2.9332e-01,  1.2019e+00,
         -9.1228e-02,  2.0432e+00, -2.6557e-03],
        [-2.2352e-01,  3.9659e-01,  4.9041e-02,  4.4417e-01,  3.3399e-01,
          4.0058e-01,  5.4472e-01, -2.9859e-01],
        [-2.6922e-01,  9.7552e-01, -5.0237e-01, -8.0897e-02,  8.0913e-02,
          6.1206e-02,  1.4597e+00,  3.4633e-01],
        [-8.3607e-01,  1.8730e-01,  2.6270e-01,  2.3950e-01,  5.7344e-01,
          2.7859e-01,  1.0056e+00, -1.2212e-01],
        [ 8.7232e-01,  9.3438e-01, -4.3098e-01, -4.0524e-01,  5.5065e-03,
         -1.6940e-01,  4.5968e-01,  3.0442e-01],
        [-5.8649e-01, -4.0352e-01,  8.0109e-01,  5.1052e-01,  9.3862e-01,
          2.4600e-01,  9.8702e-02, -5.9826e-01],
        [ 1.7241

## Normalization and Residual Additions

In transformer models, after the attention computation, the result is added with the input (this allows for better gradients), and normalized.

The normalization step is done using Layer Normalization.

In [11]:
class LayerNorm(nn.Module):

    def __init__(self, normalized_shape: int, eps: float = 1e-6):
        super().__init__()

        self.normalized_shape = normalized_shape
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=True)  # Use unbiased variance
        std = (var + self.eps).sqrt()
        
        # Normalize, scale, and shift
        return self.gamma * (x - mean) / (std + self.eps) + self.beta
        



## Multi Layer Perceptron (MLP)

After the attention step in transformers, each embedding is put through a feed forward neural network consisting of 2 fully connected linear layers with an activation in between, also known as a multilayer perceptron. 

In [12]:

class MLP(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, activation: Optional[nn.Module]=None):
        super().__init__()
        self.activation = activation if activation is not None else F.gelu
        self.layer1 = nn.Linear(in_features=d_model, out_features=d_ff)
        self.layer2 = nn.Linear(in_features=d_ff, out_features=d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of shape (batch_size, sequence_length, d_model)

        returns a tensor of shape (batch_size, sequence_length, d_model)
        '''

        return self.layer2(self.activation(self.layer1(x)))


## Putting a Transformer Together

Here we put together a transformer encoder and decoder block, complete with attention, FFN, and normalization. One trick that is used for better training is dropout, where neurons are randomly set to 0 during training. This allows the network to be better at generalizing to the dataset, and prevent overfitting. We can put a number of these blocks in sequence to form a complete transformer model.

In [13]:

class TransformerBlock(nn.Module):

    def __init__(self, **kwargs):
        '''
        **kwargs:
            d_model: int (default: 512) - The dimension of the model's hidden states, or embeddings
            d_k: int (default: 64) - The dimension of the key and value vectors in each head
            heads: int (default: 8) - The number of heads in the multiheaded attention
            d_ff: int (default: 2048) - The hidden dimension of the MLP
            dropout: float (default: 0.1) - The dropout rate
            decoder: bool (default: True) - Set to True for masked self-attention (in decoder models), otherwise set to False
        '''
        super().__init__()

        # kwargs is a dictionary. the "get" method tries to retrive the entry with a given key from a dict,
        # and returns an optional default value
        self.d_model = kwargs.get("d_model", 512) 
        self.d_k = kwargs.get("d_k", 64)
        self.heads = kwargs.get("heads", 8)
        self.d_ff = kwargs.get("d_ff", 2048)
        self.dropout = kwargs.get("dropout", 0.1)
        self.decoder = kwargs.get("decoder", True)
        self.pos_enc = kwargs.get("pos_enc", "rope")

        self.layer_norm1 = LayerNorm(normalized_shape=self.d_model)
        self.layer_norm2 = LayerNorm(normalized_shape=self.d_model)
        
        self.attention = MultiHeadAttention(
            d_model=self.d_model,
            d_k=self.d_k,
            d_v=self.d_k,
            heads=self.heads,
            decoder=self.decoder,
            pos_enc=self.pos_enc
        )

        self.mlp = MLP(d_model=self.d_model, d_ff=self.d_ff)

        self.dropout1 = nn.Dropout(p=self.dropout)
        self.dropout2 = nn.Dropout(p=self.dropout)    
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of shape (batch_size, sequence_length, d_model)

        returns a tensor of shape (batch_size, sequence_length, d_model)
        '''

        # Pre norm (deviates from original paper, but supposed to be better)
        residual = x
        x = self.layer_norm1(x)
        x = self.attention(x)
        x = self.dropout1(x) + residual

        residual = x
        x = self.layer_norm2(x)
        x = self.mlp(x)
        x = self.dropout2(x) + residual

        return x

# this takes the output of the last decoder block and passes it through a linear layer, which is supposed to be softmaxed.
class LinearHead(nn.Module):
    '''
    A linear head for various tasks including next token prediction
    '''

    def __init__(self, **kwargs):
        super().__init__()

        self.d_model = kwargs.get("d_model", 512)
        
        self.vocab_size = kwargs.get("vocab_size")
        if self.vocab_size is None:
            raise ValueError("vocab_size must be specified")

        self.linear = nn.Linear(in_features=self.d_model, out_features=self.vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = self.linear(x)
        return x
        


class Transformer(nn.Module):
    '''
    A full implementation of either a Transformer encoder or decoder model.
    '''

    def __init__(
        self,
        d_model: int = 512,
        d_k: int = 64,
        heads: int = 8,
        d_ff: int = 2048,
        dropout: float = 0.1,
        decoder: bool = True,
        num_blocks: int = 6,
        vocab_size: int = None,
        max_seq_len: int = 512,
        lm_head: nn.Module = None,
        pos_enc: str = "rope",
    ):
        '''
        Arguments:
            d_model: int (default: 512) - The dimension of the model's hidden states, or embeddings
            d_k: int (default: 64) - The dimension of the key and value vectors in each head
            heads: int (default: 8) - The number of heads in the multiheaded attention
            d_ff: int (default: 2048) - The hidden dimension of the MLP
            dropout: float (default: 0.1) - The dropout rate
            decoder: bool (default: True) - Set to True for masked self-attention (in decoder models), otherwise set to False
            num_blocks: int (default: 6) - The number of transformer blocks in the model
            vocab_size: int - The size of the vocabulary
            max_seq_len: int (default: 512) - The maximum sequence length
            lm_head: nn.Module - The linear head for the model
            pos_enc: str - The type of positional encoding to use (either "rope" or "learned")
        '''
        arguments = locals()
        arguments.pop("self")

        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.heads = heads
        self.d_ff = d_ff
        self.dropout = dropout
        self.decoder = decoder
        self.num_blocks = num_blocks
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.lm_head = lm_head
        if self.lm_head is None:
            self.lm_head = LinearHead(d_model=self.d_model, vocab_size=self.vocab_size)

        self.pos_enc = pos_enc
        if self.pos_enc != "rope" and self.pos_enc != "learned":
            raise ValueError("pos_enc must be either 'rope' or 'learned'")
        
        if self.vocab_size is None:
            raise ValueError("vocab_size must be specified")

        # maps token ids to their embeddings
        self.embed = nn.Embedding(self.vocab_size, self.d_model)

        # positional encoding
        if self.pos_enc == "rope":
            self.pe = None
        elif self.pos_enc == "learned":
            self.pe = LearnedPositionalEncoding(self.d_model, self.max_seq_len)

        self.blocks = nn.ModuleList([TransformerBlock(**arguments) for _ in range(self.num_blocks)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of size (batch_size, seq_len) of token ids
        '''

        # Maps token ids to their embeddings
        x = self.embed(x)

        # Add positional encodings if using learned positional encodings
        if self.pe is not None:
            x = self.pe(x)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Pass through linear head
        x = self.lm_head(x)

        return x





## Training

To train a decoder only model (GPT), we create a torch dataset and train it on **causal language modeling**. The loss function is CrossEntropyLoss, which measures how far the probability distribution differs from predicting the right token at each step. The labels represent the next token, and for the last token, there is no label (so we append an unused token).

In [15]:


from torch.nn.utils.rnn import pad_sequence
import pandas as pd

class TextDataset(Dataset):

    def __init__(self, tokenizer=None, **kwargs):

        self.tokenizer = tokenizer if tokenizer is not None else CharacterTokenizer()
        self.text_df = kwargs["text"]
        self.max_seq_len = kwargs.get("max_seq_len", 512)
        self.pad_token = kwargs.get("pad_token", -1)

    def __len__(self):
        return len(self.text_df)

    def __getitem__(self, index):

        text = self.text_df[index][:self.max_seq_len]
        tokens = self.tokenizer.tokenize(text)[:self.max_seq_len]
        labels = tokens[1:]

        labels = torch.cat((labels, torch.tensor([self.pad_token])), dim=-1)
        return tokens, labels

def collate_fn(batch, pad_token = -1):
    sequences, targets = zip(*batch)
    return pad_sequence(sequences, batch_first=True, padding_value=0), pad_sequence(targets, batch_first=True, padding_value=pad_token)


In [17]:
%pip install datasets
from datasets import load_dataset

tiny_stories_df = load_dataset("roneneldan/TinyStories", split="train").to_pandas()

Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 548227.78 examples/s]


Now we write the training loop for the model. We will use pandas to open the dataset.

In [18]:

import gc

torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = Transformer(
    dropout=0.1,
    decoder=True,
    num_blocks=6,
    vocab_size=256,
    pos_enc="rope"
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

dataset = TextDataset(
    tokenizer=CharacterTokenizer(),
    text=tiny_stories_df["text"],
)


optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-1)

train_loader = DataLoader(dataset=dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

epochs = 1

for epoch in range(epochs):
    batches = 0
    total = 0
    for i, (batch, labels) in enumerate(train_loader):
        optimizer.zero_grad()

        batch = batch.to(device)
        labels = labels.to(device)

        output = model(batch)
        loss = criterion(output.view(-1, output.size(-1)), labels.view(-1))

        loss.backward()
        optimizer.step()

        total += loss.item()
        batches += 1

        if i % 20 == 0:
            print(f"Epoch {epoch}, Batch {i}, Loss {total / batches}")
            total = 0
            batches = 0

Using device: cpu
Total parameters: 19167488


KeyboardInterrupt: 

## Evaluation

Some qualitative tests (sampling some text from the transformer model).

In [ ]:

# Generate a sample text

tokens = torch.tensor([79], dtype=torch.long).unsqueeze(0).to(device)
model.eval()

for x in range(100):

    with torch.no_grad():  # No gradients needed during inference

        logits = model(tokens)[:, -1, :]  # Use model() instead of model.forward()

        print(torch.linalg.vector_norm(logits))  # Debugging output

        temperature = 0.8  # Adjust for better sampling stability
        probs = F.softmax(logits / temperature, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat((tokens, next_token.detach()), dim=-1)  # Detach next token

print(CharacterTokenizer().decode(tokens[0].tolist()))

## Saving the model

In [ ]:

torch.save(model.state_dict(), "model.pt")

## Load a Model

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(
    dropout=0.1,
    decoder=True,
    num_blocks=6,
    vocab_size=256
).to(device)
model.load_state_dict(torch.load("model.pt"))


In [ ]:

# Generate a sample text

tokenizer = CharacterTokenizer()
tokens = tokenizer.tokenize("O").\
                    to(device).\
                    unsqueeze(0)
model.eval()

for x in range(512 - len(tokens[0])):

    with torch.no_grad():  # No gradients needed during inference

        logits = model(tokens)[:, -1, :]  # Use model() instead of model.forward()

        temperature = 0.8  # Adjust for better sampling stability
        probs = F.softmax(logits / temperature, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat((tokens, next_token.detach()), dim=-1)  # Detach next token

print(CharacterTokenizer().decode(tokens[0].tolist()))